In [25]:
import time
import pickle
import math

In [26]:
%load_ext autoreload
%autoreload 2
import os
import sys
module_path = os.path.abspath(os.path.join('..')) 
sys.path.insert(0, module_path)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
from hex_plot import plot_problem
from hex_star import PathfindingProblem, f, time_to_goal, best_first_search

In [28]:
maps_dir = "maps"

In [29]:
def save_layout(layout, filename):
    with open(filename, 'wb') as file:
        pickle.dump(layout, file)

def load_layout(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

In [39]:
def solve(filename):
    hex_layout = load_layout(filename)
    state_tuple = (
        hex_layout['agent'],
        hex_layout['velocity']
    )
    
    params = {
        'hex_map': hex_layout['hex_map'],
        'obstacle_map': hex_layout['obstacle_map'],
        'goal_loc': hex_layout['goal'],
        'hex_radius': hex_layout['hex_radius'],
        'hex_size': hex_layout['hex_size'],
        'acceleration_max': 5,
        'deceleration_max': 5,
        'lat_acceleration_max': 2
    }
    params['q_learning_rate'] = 0.01
    params['q_discount_factor'] = 1
    params['agent_size_r'] = 1
    h = time_to_goal
    problem = PathfindingProblem(state_tuple, **params)
    solution = best_first_search(problem, f, h)
    return problem, solution
    

In [40]:
def get_solution_path(solution, nodes=True):
    out = []
    current_node = solution
    while current_node is not None:
        loc, v = current_node.state
        if nodes:
            out.append(current_node)
        else:
            out.append(loc)
        current_node = current_node.parent
    return list(reversed(out))

In [41]:
# Dictionaries to store the data
states_data = {}
nodes_data = {}
branching_data = {}
depths_data = {}
problems = {}
solutions = {}

# Heuristic functions to use
#heuristics = [zero_heuristic, box_goal_euclidian_distance, box_goal_manhattan_distance]

maps = [
    #'r3h0.33.pkl',
    'r15h1.00.pkl',
    #'r30h5.00.pkl',
    #'r40h5.00.pkl',
    #'r50h5.00.pkl'
]

for m in maps:
    print(f"Processing map file: {m}\n")
    states_expanded = []
    nodes_generated = []
    eff_branching_factors = []
    depths = []
    
        

    # Start the timer
    start = time.time()

    # Execute the search
    problem, solution = solve(maps_dir +"/"+ m)

    # Use the best solution
    solution = solution[0]

    # Stop the timer
    end = time.time()

    # If a solution was found
    if solution is not None:
        consistent, states, nodes = problem.get_benchmarks()
        solution_cost = solution.path_cost
        states_expanded.append(states)
        nodes_generated.append(nodes)
        depths.append(solution_cost)

        current_node = solution
        solution_path = get_solution_path(solution, nodes=True)
        
            
        
        #velocities = list(reversed(velocities))
        #path_costs = list(reversed(path_costs))
        eff_branch_factor = pow(nodes, 1/len(solution_path))
        eff_branching_factors.append(eff_branch_factor)
        plot_problem(problem, solution)
    else:
        solution_cost = None
        eff_branch_factor = None
    print(f"Solution depth {len(solution_path)} found in {end-start} seconds\nExplored {nodes} states.")
    if not consistent:
        print(f"Test case {m} has shown the heuristic is not consistent")
    states_data[m] = states_expanded
    nodes_data[m] = nodes_generated
    branching_data[m] = eff_branching_factors
    depths_data[m] = depths
    problems[m] = problem
    solutions[m] = solution

Processing map file: r15h1.00.pkl



TypeError: unsupported operand type(s) for &: 'set' and 'list'

In [ ]:
states_data

In [ ]:
nodes_data

In [ ]:
branching_data

In [ ]:
depths_data